### Chains with multiple inputs

In [1]:
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

True

In [1]:
from langchain.prompts.prompt import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain.chains.llm import LLMChain

llm = ChatOpenAI(model="gpt-4o-mini")

prompt_template = PromptTemplate(input_variables=["input"], template="Tell me a joke about {input}")
chain = LLMChain(llm=llm, prompt=prompt_template)
chain.invoke(input="a parrot")

/Users/nicyscaria/Documents/GitHub/da-225o-deep-learning/da225o/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/_v/9gq712613816czhrgk8ry0tw0000gp/T/ipykernel_9449/2486307642.py:8: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  chain = LLMChain(llm=llm, prompt=prompt_template)


{'input': 'a parrot',
 'text': 'Why did the parrot wear a raincoat?\n\nBecause it wanted to be a polyunsaturated!'}

In [4]:
prompt_template = PromptTemplate(input_variables=["input", "language"], template="Tell me a joke about {input} in {language}")
chain = LLMChain(llm=llm, prompt=prompt_template)
chain.invoke({"input": "a parrot", "language": "Hindi"})

{'input': 'a parrot',
 'language': 'Hindi',
 'text': 'एक आदमी ने एक तोते को खरीदा। उसने तोते से कहा, "कुछ बोलो।"\n\nतोता बोला, "तू रोटी खा, मैं चटनी लाऊं!"\n\nआदमी हैरान होकर बोला, "तू बात मत बना, क्या तुम इंसान हो?"\n\nतोता चुटकी लेते हुए बोला, "नहीं, मैं तोता हूं, पर रोटी तो इंसान जैसा ही खा सकता हूं!" 😂🐦'}

Chains can be more complex and not all sequential chains will be as simple as passing a single string as an argument and getting a single string as output for all steps in the chain

In [2]:
from langchain.chains.sequential import SequentialChain

# This is an LLMChain to write a review given a dish name and the experience.
prompt_review = PromptTemplate.from_template(
    template="You ordered {dish_name} and your experience was {experience}. Write a review: "
)
chain_review = LLMChain(llm=llm, prompt=prompt_review, output_key="review")

# This is an LLMChain to write a follow-up comment given the restaurant review.
prompt_comment = PromptTemplate.from_template(
    template="Given the restaurant review: {review}, write a follow-up comment: "
)
chain_comment = LLMChain(llm=llm, prompt=prompt_comment, output_key="comment")

# This is an LLMChain to summarize a review.
prompt_summary = PromptTemplate.from_template(
    template="Summarise the review in one short sentence: \n\n {comment}"
)
chain_summary = LLMChain(llm=llm, prompt=prompt_summary, output_key="summary")

# This is an LLMChain to translate a summary into German.
prompt_translation = PromptTemplate.from_template(
    template="Translate the summary to Hindi: \n\n {summary}"
)
chain_translation = LLMChain(
    llm=llm, prompt=prompt_translation, output_key="hindi_translation"
)

In [3]:
overall_chain = SequentialChain(
    chains=[chain_review, chain_comment, chain_summary, chain_translation],
    input_variables=["dish_name", "experience"],
    output_variables=["review", "comment", "summary", "hindi_translation"],
)

overall_chain.invoke({"dish_name": "Pizza Salami", "experience": "It was awful!"})

{'dish_name': 'Pizza Salami',
 'experience': 'It was awful!',
 'review': "Title: Disappointing Experience with Pizza Salami\n\nI recently ordered a Pizza Salami, and I must say, it was one of the worst dining experiences I've had in a long time. \n\nFirstly, the delivery took far longer than anticipated. After waiting for over an hour, I was eagerly anticipating a delicious pizza, only to have my excitement crushed upon opening the box. The smell was off, and the presentation was far from appetizing. The crust was soggy and lacked the golden-brown finish that you would expect from a decent pizzeria.\n\nNow, let's talk about the toppings. The salami was dry and tasted stale, almost as if it had been sitting out for too long. Instead of the savory, robust flavor I had hoped for, it was bland and underwhelming. To top it off, there was a serious lack of cheese – hardly any melted goodness to complement the toppings, which contributed to an overall dry texture. \n\nThe sauce, if you could 

### Instead of chaining multiple chains together we can also use an LLM to decide which follow up chain is being used

In [4]:
from langchain.chains.router import MultiPromptChain
from langchain.chains.llm import LLMChain
from langchain.prompts import PromptTemplate
from langchain.chains.router.llm_router import LLMRouterChain, RouterOutputParser
from langchain.chains.router.multi_prompt_prompt import MULTI_PROMPT_ROUTER_TEMPLATE

positive_template = """You are an AI that focuses on the positive side of things. \
Whenever you analyze a text, you look for the positive aspects and highlight them. \
Here is the text:
{input}"""

neutral_template = """You are an AI that has a neutral perspective. You just provide a balanced analysis of the text, \
not favoring any positive or negative aspects. Here is the text:
{input}"""

negative_template = """You are an AI that is designed to find the negative aspects in a text. \
You analyze a text and show the potential downsides. Here is the text:
{input}"""

In [5]:
prompt_infos = [
    {
        "name": "positive",
        "description": "Good for analyzing positive sentiments",
        "prompt_template": positive_template,
    },
    {
        "name": "neutral",
        "description": "Good for analyzing neutral sentiments",
        "prompt_template": neutral_template,
    },
    {
        "name": "negative",
        "description": "Good for analyzing negative sentiments",
        "prompt_template": negative_template,
    },
]

In [9]:
destination_chains = {}
for p_info in prompt_infos:
    # Here we are creating a dictionary of chains, where the key is the name of the chain and the value is the chain itself.
    name = p_info["name"]
    prompt_template = p_info["prompt_template"]
    prompt = PromptTemplate(template=prompt_template, input_variables=["input"])
    chain = LLMChain(llm=llm, prompt=prompt)
    destination_chains[name] = chain
destination_chains

{'positive': LLMChain(verbose=False, prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='You are an AI that focuses on the positive side of things. Whenever you analyze a text, you look for the positive aspects and highlight them. Here is the text:\n{input}'), llm=ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x117f44670>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x117f44a30>, root_client=<openai.OpenAI object at 0x117f44370>, root_async_client=<openai.AsyncOpenAI object at 0x121563fd0>, model_name='gpt-4o-mini', model_kwargs={}, openai_api_key=SecretStr('**********'), stream_usage=True), output_parser=StrOutputParser(), llm_kwargs={}),
 'neutral': LLMChain(verbose=False, prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='You are an AI that has a neutral perspective. You just provide a balanced analysis of the 

In [10]:
destinations = [f"{p['name']}: {p['description']}" for p in prompt_infos]
destinations_str = "\n".join(destinations)
print(destinations_str)

positive: Good for analyzing positive sentiments
neutral: Good for analyzing neutral sentiments
negative: Good for analyzing negative sentiments


In [11]:
router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(destinations=destinations_str)
router_prompt = PromptTemplate(
    template=router_template,
    input_variables=["input"],
    output_parser=RouterOutputParser(), # RouterOutputParser is used to parse the output of the router chain into a dictionary
)

router_chain = LLMRouterChain.from_llm(llm, router_prompt)

chain = MultiPromptChain(
    router_chain=router_chain,
    destination_chains=destination_chains,
    default_chain=destination_chains["neutral"],
    verbose=True,
)

chain.invoke({"input": "I ordered Pizza Salami for 9.99$ and it was not good!"})

/var/folders/_v/9gq712613816czhrgk8ry0tw0000gp/T/ipykernel_10830/731848241.py:10: LangChainDeprecationWarning: Please see migration guide here for recommended implementation: https://python.langchain.com/docs/versions/migrating_chains/multi_prompt_chain/
  chain = MultiPromptChain(




> Entering new MultiPromptChain chain...
negative: {'input': 'I ordered Pizza Salami for $9.99 and it was not good!'}
> Finished chain.


{'input': 'I ordered Pizza Salami for $9.99 and it was not good!',
 'text': 'The potential downsides in the provided text include:\n\n1. **Quality Issues**: The phrase "it was not good" indicates a significant dissatisfaction with the pizza quality, suggesting that the taste or preparation did not meet expectations.\n\n2. **Disappointment**: The mention of ordering something and then not being satisfied reflects a negative experience, potentially leading to frustration for the customer.\n\n3. **Money Wasted**: Spending $9.99 on a product that did not meet standards can feel like a waste of money, contributing to feelings of loss or regret.\n\n4. **Limited Value**: The price may imply a certain level of quality, and failing to deliver on that value could cause customers to reconsider future purchases.\n\n5. **Potential for Negative Reviews**: Such dissatisfaction may lead the customer to share their experience with others, potentially harming the restaurant\'s reputation.\n\n6. **Lack o